# Part 3 : Arbres de Décision & Random Forest

**Durée estimée : 3h**

## Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Expliquer** l'intuition d'un arbre de décision (série de questions if/else)
2. **Comprendre** comment l'arbre choisit ses questions (critère de Gini)
3. **Maîtriser** les hyperparamètres pour contrôler la complexité
4. **Diagnostiquer** l'underfitting et l'overfitting (biais-variance)
5. **Appliquer** Random Forest pour des prédictions plus robustes
6. **Analyser** l'importance des features

---

## Problème Réel : Comment une banque décide-t-elle de vous accorder un prêt ?

Vous faites une demande de crédit immobilier. En quelques jours (parfois quelques minutes en ligne !), la banque vous donne sa réponse : **approuvé** ou **refusé**.

**Question :** Comment la banque peut-elle évaluer si vous allez rembourser ou faire défaut ?

*(Prenez 30 secondes pour réfléchir...)*

<details>
<summary>Votre intuition ?</summary>

### Réponse

La banque pose une série de **questions** sur votre profil :
- Quel est votre revenu ?
- Avez-vous un emploi stable ?
- Quel est votre historique de crédit ?

Selon les réponses, elle suit un **arbre de décision** interne.

</details>

### Le scoring crédit : un cas d'étude réel

Selon une [étude 2024 (Engineering Reports)](https://onlinelibrary.wiley.com/doi/10.1002/eng2.12707), Random Forest atteint **91% de précision** dans la prédiction d'approbation de prêt, contre 83% pour un arbre simple.

| Facteur analysé | Impact sur la décision |
|-----------------|------------------------|
| Historique de crédit | Très fort |
| Ratio dette/revenu | Fort |
| Durée d'emploi | Moyen |
| Montant du prêt | Moyen |

---

## 3.1 Intuition : Une Série de Questions

Imaginez le processus de décision d'un banquier :

```
┌───────────────────────────────────────────────────────────────────────┐
│              ARBRE DE DÉCISION : Approuver un prêt ?                  │
├───────────────────────────────────────────────────────────────────────┤
│                    ┌─────────────────────┐                            │
│                    │ Revenu > 35 000€/an │                            │
│                    └──────────┬──────────┘                            │
│                       OUI /        \ NON                              │
│                          /          \                                 │
│           ┌──────────────┐          ┌──────────────┐                  │
│           │ Historique   │          │ Co-emprunteur│                  │
│           │ crédit bon ? │          │ disponible ? │                  │
│           └──────┬───────┘          └──────┬───────┘                  │
│            OUI /   \ NON             OUI /   \ NON                    │
│               /     \                   /     \                       │
│       ┌──────┐    ┌──────┐      ┌──────┐    ┌──────┐                  │
│       │  ✅  │    │Ratio │      │  ✅  │    │  ❌  │                  │
│       │APPROUV│    │dette?│      │APPROUV│    │REFUSÉ│                  │
│       └──────┘    └──┬───┘      └──────┘    └──────┘                  │
│                  <30% / ≥30%                                          │
│                     /    \                                            │
│              ┌──────┐  ┌──────┐                                       │
│              │  ✅  │  │  ❌  │                                       │
│              │APPROUV│  │REFUSÉ│                                       │
│              └──────┘  └──────┘                                       │
└───────────────────────────────────────────────────────────────────────┘
```

**Question Socratique :** Pourquoi cet arbre est-il facile à comprendre ?

*(Réponse : On peut suivre le raisonnement pas à pas. On peut même expliquer à un client pourquoi son prêt a été refusé.)*

---

## 3.2 Construction : Le Critère de Gini

### Comment l'arbre choisit ses questions ?

Imaginez un sac de billes : 🔴 (refusé) et 🟢 (approuvé).

```
┌─────────────────────────────────────────────────────────────────────┐
│     L'ANALOGIE DES BILLES                                          │
├─────────────────────────────────────────────────────────────────────┤
│  Sac initial : 🔴🔴🔴🟢🟢🔴🟢🔴🟢🟢  (mélangé)                       │
│                                                                     │
│  Question A : "Revenu > 40 000 ?"                                   │
│  ┌─────────────────┐     ┌─────────────────┐                        │
│  │ OUI : 🟢🟢🟢🟢🔴  │     │ NON : 🔴🔴🔴🔴🟢  │                        │
│  │ (80% vert)      │     │ (80% rouge)     │                        │
│  └─────────────────┘     └─────────────────┘                        │
│  → BONNE séparation ! Groupes assez "purs"                         │
│                                                                     │
│  Question B : "Statut marital ?"                                    │
│  ┌─────────────────┐     ┌─────────────────┐                        │
│  │ OUI : 🔴🟢🔴🟢🔴  │     │ NON : 🟢🔴🟢🔴🟢  │                        │
│  │ (50% chaque)    │     │ (50% chaque)    │                        │
│  └─────────────────┘     └─────────────────┘                        │
│  → MAUVAISE séparation ! Groupes restent mélangés                  │
└─────────────────────────────────────────────────────────────────────┘
```

L'arbre choisit la question qui crée les groupes les plus **purs**.

### L'indice de Gini

L'**indice de Gini** mesure l'impureté d'un groupe :

```
┌───────────────────────────────────────────────────────────────┐
│                    INDICE DE GINI                             │
├───────────────────────────────────────────────────────────────┤
│  Gini = 0          Groupe "pur" (tous de la même classe)     │
│  ●●●●●●●●●●                                                   │
│                                                               │
│  Gini = 0.5        Groupe "mélangé" (50% de chaque)          │
│  ●●●●●○○○○○                                                   │
│                                                               │
│  L'arbre cherche à MINIMISER le Gini après chaque split       │
└───────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
# Créer un dataset de prêts
np.random.seed(42)
n = 500

data = pd.DataFrame({
    'revenu': np.random.uniform(20000, 100000, n),
    'historique_credit': np.random.choice([0, 1], n, p=[0.3, 0.7]),
    'ratio_dette': np.random.uniform(0, 0.6, n),
    'anciennete_emploi': np.random.uniform(0, 20, n),
    'montant_pret': np.random.uniform(5000, 50000, n)
})

# Cible selon des règles + bruit
data['approuve'] = (
    (data['revenu'] > 40000) & 
    (data['historique_credit'] == 1) & 
    (data['ratio_dette'] < 0.4)
).astype(int)
flip_idx = np.random.choice(n, size=int(n*0.1), replace=False)
data.loc[flip_idx, 'approuve'] = 1 - data.loc[flip_idx, 'approuve']

print("Aperçu des données :")
data.head()

In [ ]:
# Split train/test
X = data.drop('approuve', axis=1)
y = data['approuve']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Entraîner un arbre (profondeur limitée)
arbre = DecisionTreeClassifier(max_depth=3, random_state=42)
arbre.fit(X_train, y_train)

# Visualiser
plt.figure(figsize=(18, 10))
plot_tree(arbre, 
          feature_names=X.columns, 
          class_names=['Refusé', 'Approuvé'],
          filled=True, rounded=True, fontsize=10)
plt.title('Arbre de Décision : Approbation de prêt', fontsize=14)
plt.tight_layout()
plt.show()

### Comment lire un nœud sklearn

```
┌─────────────────────────────┐
│  ratio_dette <= 0.405      │  ← La QUESTION
│  gini = 0.48               │  ← Pureté (0=pur, 0.5=mélangé)
│  samples = 400             │  ← Combien d'exemples
│  value = [180, 220]        │  ← [Refusé, Approuvé]
│  class = Approuvé          │  ← Prédiction majoritaire
└─────────────────────────────┘
```

---

## 3.3 Hyperparamètres : Contrôler la Complexité

**Rappel (Ch2 Part3)** : Les hyperparamètres sont des choix AVANT l'entraînement.

```
┌───────────────────────────────────────────────────────────────────────┐
│     HYPERPARAMÈTRES DE DecisionTreeClassifier                        │
├───────────────────────────────────────────────────────────────────────┤
│                                                                       │
│  max_depth = 5                                                        │
│  └─ Profondeur maximale de l'arbre                                    │
│     • Plus petit = arbre simple, risque d'UNDERFITTING                │
│     • Plus grand = arbre complexe, risque d'OVERFITTING               │
│                                                                       │
│  min_samples_split = 20                                               │
│  └─ Minimum d'exemples pour créer un split                            │
│     • "Si < 20 exemples, arrête de subdiviser"                        │
│                                                                       │
│  min_samples_leaf = 10                                                │
│  └─ Minimum d'exemples dans chaque feuille                            │
│     • Empêche les feuilles avec 1-2 exemples (= mémorisation)         │
│                                                                       │
│  criterion = 'gini' ou 'entropy'                                      │
│  └─ Mesure d'impureté (gini par défaut)                               │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Comparer différentes profondeurs
profondeurs = [1, 3, 5, 10, None]
resultats = []

for depth in profondeurs:
    arbre_test = DecisionTreeClassifier(max_depth=depth, random_state=42)
    arbre_test.fit(X_train, y_train)
    
    train_score = arbre_test.score(X_train, y_train)
    test_score = arbre_test.score(X_test, y_test)
    
    resultats.append({
        'max_depth': str(depth) if depth else 'None (illimité)',
        'Train': train_score,
        'Test': test_score,
        'Gap': train_score - test_score,
        'Profondeur réelle': arbre_test.get_depth()
    })

df_res = pd.DataFrame(resultats)
print("═" * 70)
print("IMPACT DE max_depth SUR LES PERFORMANCES")
print("═" * 70)
print(df_res.to_string(index=False))

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train vs Test
x = range(len(df_res))
width = 0.35
axes[0].bar([i - width/2 for i in x], df_res['Train'], width, label='Train', color='lightblue')
axes[0].bar([i + width/2 for i in x], df_res['Test'], width, label='Test', color='orange')
axes[0].set_ylabel('Accuracy')
axes[0].set_xlabel('max_depth')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['1', '3', '5', '10', 'None'])
axes[0].legend()
axes[0].set_title('Performances selon max_depth')

# Gap (signe d'overfitting)
colors = ['green' if g < 0.05 else 'orange' if g < 0.15 else 'red' for g in df_res['Gap']]
axes[1].bar(x, df_res['Gap'], color=colors)
axes[1].axhline(y=0.05, color='orange', linestyle='--', label='Seuil acceptable')
axes[1].set_ylabel('Gap (Train - Test)')
axes[1].set_xlabel('max_depth')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['1', '3', '5', '10', 'None'])
axes[1].legend()
axes[1].set_title('Gap = signe d\'overfitting')

plt.tight_layout()
plt.show()

### Résumé des hyperparamètres

| Hyperparamètre | Effet si augmenté | Risque |
|----------------|-------------------|--------|
| max_depth | Arbre plus complexe | Overfitting |
| min_samples_split | Arbre plus simple | Underfitting |
| min_samples_leaf | Arbre plus simple | Underfitting |

---

## 3.4 Évaluation : Le Compromis Biais-Variance

**Rappel (Ch2 Part3)** : Vous avez appris les métriques de classification (Accuracy, Precision, Recall, F1).

Maintenant, comprenons POURQUOI un modèle performe mal.

### L'analogie du tireur à l'arc

```
┌─────────────────────────────────────────────────────────────────────┐
│           BIAIS vs VARIANCE                                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   HAUT BIAIS (Underfitting)      HAUTE VARIANCE (Overfitting)      │
│                                                                     │
│         🎯                              🎯                          │
│        /│\                             /│\                          │
│       / │ \                           / │ \        ●                │
│      /  │  \                         /  │  \                        │
│     /   │   \                       /   │   \   ●                   │
│    /    │    \                     /    │    \                      │
│   ●     │     \               ●   /     │     \     ●               │
│   ●     │      \                 /      │      \                    │
│   ●     │       \               /       │       ●                   │
│                                                                     │
│   Tirs GROUPÉS mais           Tirs DISPERSÉS partout               │
│   loin de la cible            autour de la cible                   │
│                                                                     │
│   → Technique défaillante     → Trop sensible aux conditions       │
│   → Modèle trop SIMPLE        → Modèle trop COMPLEXE               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### Diagnostic rapide

```
┌─────────────────────────────────────────────────────────────────────┐
│           GUIDE DE DIAGNOSTIC                                       │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   OBSERVATION          │ DIAGNOSTIC      │ ACTION                   │
│   ─────────────────────┼─────────────────┼─────────────────────────│
│   Train BAS            │                 │                          │
│   Test BAS             │ UNDERFITTING    │ Augmenter max_depth      │
│   Écart FAIBLE         │ (Haut Biais)    │ Plus de features         │
│                        │                 │                          │
│   ─────────────────────┼─────────────────┼─────────────────────────│
│   Train ÉLEVÉ (≈100%)  │                 │                          │
│   Test BAS             │ OVERFITTING     │ Réduire max_depth        │
│   Écart GRAND          │ (Haute Variance)│ min_samples_leaf ↑       │
│                        │                 │ Random Forest            │
│   ─────────────────────┼─────────────────┼─────────────────────────│
│   Train ÉLEVÉ          │                 │                          │
│   Test ÉLEVÉ           │ BON ÉQUILIBRE   │ Continuer !             │
│   Écart FAIBLE         │ ✓               │                          │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Fonction d'évaluation de Ch2 Part3
def evaluer_classification(y_vrai, y_pred, nom_modele="Modèle"):
    """Affiche toutes les métriques de classification."""
    print(f"\nÉvaluation : {nom_modele}")
    print("=" * 50)
    print(f"Accuracy  : {accuracy_score(y_vrai, y_pred):.4f}")
    print(f"Precision : {precision_score(y_vrai, y_pred, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_vrai, y_pred, zero_division=0):.4f}")
    print(f"F1-Score  : {f1_score(y_vrai, y_pred, zero_division=0):.4f}")
    
    return {
        'Accuracy': accuracy_score(y_vrai, y_pred),
        'Precision': precision_score(y_vrai, y_pred, zero_division=0),
        'Recall': recall_score(y_vrai, y_pred, zero_division=0),
        'F1': f1_score(y_vrai, y_pred, zero_division=0)
    }

In [ ]:
# Comparer arbre simple vs arbre profond
arbre_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
arbre_simple.fit(X_train, y_train)

arbre_profond = DecisionTreeClassifier(max_depth=None, random_state=42)
arbre_profond.fit(X_train, y_train)

print("═" * 60)
print("DIAGNOSTIC BIAIS-VARIANCE")
print("═" * 60)

for nom, modele in [('Arbre Simple (depth=3)', arbre_simple), 
                     ('Arbre Profond (unlimited)', arbre_profond)]:
    train_acc = modele.score(X_train, y_train)
    test_acc = modele.score(X_test, y_test)
    gap = train_acc - test_acc
    
    print(f"\n{nom}:")
    print(f"  Train: {train_acc:.1%} | Test: {test_acc:.1%} | Gap: {gap:.1%}")
    
    if train_acc < 0.75 and test_acc < 0.75:
        print("  → UNDERFITTING (haut biais) : augmenter la complexité")
    elif gap > 0.15:
        print("  → OVERFITTING (haute variance) : simplifier ou Random Forest")
    else:
        print("  → BON ÉQUILIBRE ✓")

### Courbes d'apprentissage (Learning Curves)

Un outil puissant pour visualiser biais vs variance.

In [ ]:
# Learning curves
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (nom, modele) in zip(axes, [('Arbre Simple (depth=3)', 
                                       DecisionTreeClassifier(max_depth=3, random_state=42)),
                                      ('Arbre Profond (unlimited)', 
                                       DecisionTreeClassifier(max_depth=None, random_state=42))]):
    train_sizes, train_scores, test_scores = learning_curve(
        modele, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10), random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    test_mean = test_scores.mean(axis=1)
    
    ax.plot(train_sizes, train_mean, 'o-', label='Train', color='blue')
    ax.plot(train_sizes, test_mean, 'o-', label='Test', color='orange')
    ax.fill_between(train_sizes, train_mean - train_scores.std(axis=1),
                    train_mean + train_scores.std(axis=1), alpha=0.1, color='blue')
    ax.fill_between(train_sizes, test_mean - test_scores.std(axis=1),
                    test_mean + test_scores.std(axis=1), alpha=0.1, color='orange')
    
    ax.set_xlabel('Taille du training set')
    ax.set_ylabel('Score')
    ax.set_title(nom)
    ax.legend(loc='lower right')
    ax.set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

print("\n📊 Interprétation :")
print("• Arbre Simple : courbes proches = pas d'overfitting, mais plafonne")
print("• Arbre Profond : grand écart = OVERFITTING (mémorise les données)")

---

## 3.5 Random Forest : La Force de l'Ensemble

**Solution à l'overfitting des arbres** : demander l'avis de BEAUCOUP d'arbres !

```
┌───────────────────────────────────────────────────────────────────────┐
│                       RANDOM FOREST                                   │
├───────────────────────────────────────────────────────────────────────┤
│                                                                       │
│    Données      ┌──────┐      ┌──────┐      ┌──────┐                  │
│       │         │Arbre1│      │Arbre2│      │Arbre3│   ... (100+)     │
│       │         │  🌲  │      │  🌲  │      │  🌲  │                  │
│       ▼         └──┬───┘      └──┬───┘      └──┬───┘                  │
│  ┌────────┐        │              │              │                    │
│  │Bootstrap│       │              │              │                    │
│  │(échant.)│       ▼              ▼              ▼                    │
│  └────────┘   "Approuvé"    "Refusé"      "Approuvé"                 │
│                    │              │              │                    │
│                    └──────────────┼──────────────┘                    │
│                                   ▼                                   │
│                           ┌──────────────┐                            │
│                           │ VOTE MAJORITÉ│                            │
│                           │  2 vs 1      │                            │
│                           │  → APPROUVÉ  │                            │
│                           └──────────────┘                            │
│                                                                       │
│  Chaque arbre voit :                                                  │
│  • Un échantillon DIFFÉRENT (bootstrap)                               │
│  • Un sous-ensemble ALÉATOIRE des features                            │
│                                                                       │
└───────────────────────────────────────────────────────────────────────┘
```

**Pourquoi ça marche ?** Chaque arbre fait des erreurs DIFFÉRENTES. En votant, les erreurs s'annulent !

### Hyperparamètres de Random Forest

| Paramètre | Défaut | Description |
|-----------|--------|-------------|
| n_estimators | 100 | Nombre d'arbres (plus = mieux, mais plus lent) |
| max_depth | None | Profondeur max de chaque arbre |
| min_samples_split | 2 | Min exemples pour split |
| max_features | 'sqrt' | Features considérées à chaque split |

In [ ]:
# Entraîner Random Forest
foret = RandomForestClassifier(n_estimators=100, random_state=42)
foret.fit(X_train, y_train)

# Comparer les trois modèles
modeles = {
    'Arbre Simple (depth=3)': arbre_simple,
    'Arbre Profond (unlimited)': arbre_profond,
    'Random Forest (100 arbres)': foret
}

print("═" * 65)
print("COMPARAISON FINALE")
print("═" * 65)

resultats_final = []
for nom, model in modeles.items():
    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    gap = train_score - test_score
    resultats_final.append({'Modèle': nom, 'Train': train_score, 'Test': test_score, 'Gap': gap})
    print(f"\n{nom}:")
    print(f"  Train: {train_score:.1%} | Test: {test_score:.1%} | Gap: {gap:.1%}")

In [ ]:
# Visualisation comparative
df_final = pd.DataFrame(resultats_final)

plt.figure(figsize=(12, 5))
x = np.arange(len(df_final))
width = 0.35

plt.bar(x - width/2, df_final['Train'], width, label='Train', color='lightblue')
plt.bar(x + width/2, df_final['Test'], width, label='Test', color='orange')

plt.ylabel('Accuracy')
plt.title('Random Forest : le meilleur des deux mondes')
plt.xticks(x, ['Arbre\nSimple', 'Arbre\nProfond', 'Random\nForest'])
plt.legend()
plt.ylim(0, 1.1)

for i, row in df_final.iterrows():
    plt.annotate(f"{row['Train']:.0%}", xy=(i - width/2, row['Train'] + 0.02), ha='center')
    plt.annotate(f"{row['Test']:.0%}", xy=(i + width/2, row['Test'] + 0.02), ha='center')

plt.tight_layout()
plt.show()

print("\n✅ Random Forest : performance élevée + faible overfitting !")

---

## 3.6 Feature Importance : Quelles Variables Comptent ?

Un avantage majeur de Random Forest : identifier les features les plus importantes.

In [ ]:
# Importance des features
importances = foret.feature_importances_
indices = np.argsort(importances)[::-1]

print("═" * 50)
print("IMPORTANCE DES FEATURES (Random Forest)")
print("═" * 50)

for i, idx in enumerate(indices):
    bars = "█" * int(importances[idx] * 50)
    print(f"  {X.columns[idx]:20s} : {importances[idx]:.1%} {bars}")

# Visualisation
plt.figure(figsize=(10, 6))
plt.barh(range(len(importances)), importances[indices], align='center')
plt.yticks(range(len(importances)), X.columns[indices])
plt.xlabel('Importance')
plt.title('Quelles features influencent le plus la décision ?')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Quand utiliser quoi ?

| Situation | Recommandation |
|-----------|----------------|
| Besoin d'expliquer la décision | Arbre simple (max_depth limité) |
| Priorité à la performance | Random Forest |
| Données avec beaucoup de bruit | Random Forest (plus robuste) |
| Identifier les features importantes | Random Forest + feature_importances_ |

---

## Exercice Pratique : Titanic

Prédire qui a survécu au naufrage.

In [ ]:
# Dataset Titanic simplifié
np.random.seed(42)
n = 800

titanic = pd.DataFrame({
    'classe': np.random.choice([1, 2, 3], n, p=[0.2, 0.3, 0.5]),
    'age': np.random.uniform(1, 80, n),
    'sexe': np.random.choice([0, 1], n),  # 0=homme, 1=femme
    'famille_a_bord': np.random.randint(0, 6, n),
    'tarif': np.random.uniform(5, 500, n)
})

# Survie (femmes et enfants d'abord)
survie_proba = 0.3*(titanic['sexe']==1) + 0.2*(titanic['age']<15) + 0.2*(titanic['classe']==1) + 0.1
titanic['survie'] = (np.random.random(n) < survie_proba).astype(int)

print(f"Taux de survie : {titanic['survie'].mean():.1%}")
titanic.head()

### Mission :
1. Split train/test
2. Entraîner un arbre (max_depth=5) et un Random Forest
3. Évaluer avec evaluer_classification()
4. Diagnostiquer : overfitting ou underfitting ?
5. Quelle feature est la plus importante ?

In [ ]:
# À VOUS DE JOUER !

# X_titanic = ...
# y_titanic = ...

# Split
# ...

# Entraîner
# ...

# Évaluer
# ...

In [ ]:
# SOLUTION

X_titanic = titanic.drop('survie', axis=1)
y_titanic = titanic['survie']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic, test_size=0.2, random_state=42
)

# Entraîner
arbre_t = DecisionTreeClassifier(max_depth=5, random_state=42)
arbre_t.fit(X_train_t, y_train_t)

foret_t = RandomForestClassifier(n_estimators=100, random_state=42)
foret_t.fit(X_train_t, y_train_t)

# Évaluer
print("═" * 55)
print("RÉSULTATS TITANIC")
print("═" * 55)

for nom, model in [('Arbre (depth=5)', arbre_t), ('Random Forest', foret_t)]:
    y_pred = model.predict(X_test_t)
    evaluer_classification(y_test_t, y_pred, nom)
    print(f"  Gap train-test: {model.score(X_train_t, y_train_t) - model.score(X_test_t, y_test_t):.1%}")

# Feature importance
print("\n" + "═" * 55)
print("FEATURES LES PLUS IMPORTANTES")
print("═" * 55)
for feat, imp in sorted(zip(X_titanic.columns, foret_t.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat:15s} : {imp:.1%}")

---

## Récapitulatif

### Structure de cette partie :

| Section | Contenu |
|---------|--------|
| **Hook** | Approbation de prêt bancaire |
| **3.1 Intuition** | Série de questions if/else |
| **3.2 Construction** | Critère de Gini, lecture d'un nœud |
| **3.3 Hyperparamètres** | max_depth, min_samples_split, min_samples_leaf |
| **3.4 Évaluation** | Biais-variance, learning curves, diagnostic |
| **3.5 Random Forest** | Ensemble d'arbres, vote majoritaire |
| **3.6 Feature Importance** | Quelles variables comptent |

### Code essentiel :

```python
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Arbre (contrôler la complexité !)
arbre = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10)
arbre.fit(X_train, y_train)

# Random Forest (robuste)
foret = RandomForestClassifier(n_estimators=100)
foret.fit(X_train, y_train)

# Feature importance
print(foret.feature_importances_)
```

### Prochaine partie : K-Means Clustering

Et si on n'avait pas de labels ? Comment trouver des groupes naturels dans les données ?

---

## Réflexion Métacognitive

1. Comment diagnostiquez-vous l'overfitting avec les scores train/test ?
2. Pourquoi Random Forest est-il plus robuste qu'un seul arbre ?
3. Quand préféreriez-vous un arbre simple interprétable à un Random Forest performant ?

---

**Sources :**
- [Wiley 2024 - Random Forest for Loan Prediction](https://onlinelibrary.wiley.com/doi/10.1002/eng2.12707)
- [scikit-learn DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- [scikit-learn RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)